In [0]:
-- Create Quantexa Data Product households_all Schema metadata
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- metadata table
DROP TABLE IF EXISTS metadata;

CREATE OR REPLACE TABLE metadata ( 
	supplier STRING,
	data_product STRING,
	data_product_version STRING,
	schema_name STRING,
	schema_version STRING,
	schema_variant_name STRING,
	schema_variant_version STRING,
	instance STRING,
	instance_version STRING
 );

INSERT INTO metadata 
(
  supplier, 
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
) 
VALUES (
  'quantexa.com',
  'com.quantexa.qdp.households_all',
  '0.1',
  'households_all.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);


In [0]:
-- Create Quantexa Data Product households_all Schema
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- hub_household table
DROP TABLE IF EXISTS hub_household;

CREATE TABLE hub_household ( 
	household_id BIGINT NOT NULL GENERATED  ALWAYS AS IDENTITY  (START WITH 1 INCREMENT BY 1) COMMENT 'Household Identity',
	household_entity_id STRING,
	tennant_id BIGINT NOT NULL DEFAULT 0,
	period_start TIMESTAMP,
	period_end TIMESTAMP,
	CONSTRAINT pk_locationall_hubhousehold_householdid PRIMARY KEY ( household_id )
 )     USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');


ALTER TABLE hub_household ALTER COLUMN household_id comment 'Household Identity';


In [0]:
-- Create Quantexa Data Product households_all Schema
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- sat_household  table
DROP TABLE IF EXISTS sat_household;

CREATE TABLE sat_household ( 
	sat_household_id BIGINT NOT NULL GENERATED  ALWAYS AS IDENTITY  (START WITH 1 INCREMENT BY 1) COMMENT 'Household Identity Sat',
	household_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	entity_id STRING,
	address_id BIGINT,
	household_occupancy INT,
	type_code STRING,
	type_concept_id BIGINT,
	type_raw_code STRING,
	type_raw_concept_id BIGINT,
	notes STRING,
	period_start TIMESTAMP,
	period_end TIMESTAMP,
	CONSTRAINT pk_locationall_sathubhousehold_sathouseholdid PRIMARY KEY ( sat_household_id)
 )     USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

ALTER TABLE sat_household ADD CONSTRAINT fk_sathousehold_hubhousehold_householdid FOREIGN KEY ( household_id ) REFERENCES hub_household( household_id ) ON DELETE NO ACTION ON UPDATE NO ACTION;

ALTER TABLE sat_household ALTER COLUMN household_id comment 'Household Identity';


In [0]:
-- Create Quantexa Data Product households_all Schema
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

DROP TABLE IF EXISTS sat_household_member;
CREATE TABLE sat_household_member ( 
	household_member_id BIGINT NOT NULL GENERATED  ALWAYS AS IDENTITY  (START WITH 1 INCREMENT BY 1) COMMENT 'Household Member Identity',
	entity_id STRING,
	household_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	individual_id BIGINT NOT NULL,
	role_code STRING,
	role_concept_id BIGINT,
	role_raw_code STRING,
	role_raw_concept_id BIGINT,
	notes STRING,
	previous_household_id BIGINT,
	next_household_id BIGINT,
	is_currently_resident INT COMMENT 'TRUE = This Individual is currently a member of this Household\nFALSE = This Individual is not currently a member of this Household, but has been so in the past  N.B.  The period_end field contains the date this Individual stopped being a member of this Household',
	period_start TIMESTAMP,
	period_end TIMESTAMP,
	CONSTRAINT pk_locationall_sathouseholdmember_householdmemberid PRIMARY KEY ( household_member_id )
 )     USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');


ALTER TABLE sat_household_member ADD CONSTRAINT fk_locationall_sathouseholdmember_hubhousehold_householdid FOREIGN KEY ( household_id ) REFERENCES hub_household( household_id ) ON DELETE NO ACTION ON UPDATE NO ACTION;

ALTER TABLE sat_household_member ADD CONSTRAINT fk_locationall_sathouseholdmember_hubhousehold_previoushouseholdlid FOREIGN KEY ( previous_household_id ) REFERENCES hub_household( household_id ) ON DELETE NO ACTION ON UPDATE NO ACTION;

ALTER TABLE sat_household_member ADD CONSTRAINT fk_locationall_sathouseholdmember_hubhousehold_nexthouseholdlid FOREIGN KEY ( next_household_id ) REFERENCES hub_household( household_id ) ON DELETE NO ACTION ON UPDATE NO ACTION;

ALTER TABLE sat_household_member ALTER COLUMN household_member_id comment 'Household Member Identity';

ALTER TABLE sat_household_member ALTER COLUMN is_currently_resident comment 'TRUE = This Individual is currently a member of this Household
FALSE = This Individual is not currently a member of this Household, but has been so in the past  N.B.  The period_end field contains the date this Individual stopped being a member of this Household';
